In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score

# Sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import VarianceThreshold

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


In [2]:
# Carregar o dataset DoS
dos_path = Path("../data/raw/MQTT Under Attack Dataset/DoS.csv")
df = pd.read_csv(dos_path)

print(f"📊 Shape do dataset: {df.shape}")
print(f"📝 Colunas: {df.shape[1]}")
print(f"📋 Registros: {df.shape[0]}")
print(f"\n🏷️ Distribuição do Label ('type'):")
print(df['type'].value_counts())

📊 Shape do dataset: (94625, 67)
📝 Colunas: 67
📋 Registros: 94625

🏷️ Distribuição do Label ('type'):
type
normal    49111
DoS       45514
Name: count, dtype: int64


In [3]:
# Criar coluna publish_gap com NaN para todos
df['publish_gap'] = np.nan

# Calcular o intervalo apenas para linhas PUBLISH (msgtype == 3)
publish_mask = df['mqtt.msgtype'] == 3
df.loc[publish_mask, 'publish_gap'] = df.loc[publish_mask, 'frame.time_epoch'].diff()

#Utilizando foward fill para propagar último valor conhecido
df['publish_gap'] = df['publish_gap'].ffill().fillna(0)


# Verificar resultado
print(f"Registros PUBLISH: {publish_mask.sum()}")
print(f"Valores não-nulos em publish_gap: {df['publish_gap'].notna().sum()}")
print(f"\nEstatísticas do publish_gap:")
print(df['publish_gap'].describe())

Registros PUBLISH: 37863
Valores não-nulos em publish_gap: 94625

Estatísticas do publish_gap:
count    94625.000000
mean         4.097958
std         10.853330
min          0.000000
25%          0.000000
50%          0.194238
75%          2.425834
max         57.168259
Name: publish_gap, dtype: float64


In [4]:
# Criar coluna connect_gap com NaN para todos
df['connect_gap'] = np.nan

# Calcular o intervalo apenas para linhas CONNECT (msgtype == 1)
connect_mask = df['mqtt.msgtype'] == 1
df.loc[connect_mask, 'connect_gap'] = df.loc[connect_mask, 'frame.time_epoch'].diff()

# Utilizando forward fill para propagar último valor conhecido
df['connect_gap'] = df['connect_gap'].ffill().fillna(0)

# Verificar resultado
print(f"Registros CONNECT: {connect_mask.sum()}")
print(f"Valores não-nulos em connect_gap: {df['connect_gap'].notna().sum()}")
print(f"\nEstatísticas do connect_gap:")
print(df['connect_gap'].describe())

Registros CONNECT: 323
Valores não-nulos em connect_gap: 94625

Estatísticas do connect_gap:
count    94625.000000
mean         1.374774
std          3.045185
min          0.000000
25%          0.000020
50%          0.000020
75%          1.986573
max        264.861559
Name: connect_gap, dtype: float64


In [5]:
# Remover features indesejadas
colunas_remover = ['mqtt.proto_len', 'mqtt.ver']
df = df.drop(columns=[c for c in colunas_remover if c in df.columns], errors='ignore')

print(f"❌ Colunas removidas: {colunas_remover}")

# Salvar o dataset com as novas features
output_path = Path("../data/processed/DoS_with_gap_features.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"\n✅ Dataset salvo em: {output_path}")
print(f"📊 Shape final: {df.shape}")
print(f"\n🆕 Features adicionadas: 'publish_gap', 'connect_gap'")
print(f"❌ Features removidas: 'mqtt.proto_len', 'mqtt.ver'")

❌ Colunas removidas: ['mqtt.proto_len', 'mqtt.ver']

✅ Dataset salvo em: ../data/processed/DoS_with_gap_features.csv
📊 Shape final: (94625, 67)

🆕 Features adicionadas: 'publish_gap', 'connect_gap'
❌ Features removidas: 'mqtt.proto_len', 'mqtt.ver'
